# Time-Varying Potentials in THRML
## Annealing, scheduling, and dynamic bias injection

Energy-based models often need non-static energy landscapes: simulated annealing,
scheduled biases, or dynamically injected constraints. This notebook demonstrates
patterns for time-varying potentials in THRML.

**Patterns covered:**
1. Step-function schedules (discrete bias injection)
2. One-shot transitions (sudden potential change)
3. Exponential decay (bias erosion over time)
4. Persistent bias (static reference case)
5. Comparing schedule strategies: which converges fastest?

**Key finding:** Schedule shape matters more than the number of injection steps —
a counterintuitive result validated against experimental data.

In [ ]:
import jax
import jax.numpy as jnp
import jax.random
import numpy as np
import matplotlib.pyplot as plt

from thrml.block_management import Block
from thrml.block_sampling import sample_states, SamplingSchedule
from thrml.models.ising import IsingEBM, IsingSamplingProgram
from thrml.pgm import SpinNode

## Schedule Construction Utilities

Four schedule types that cover the most common use cases for time-varying potentials.

In [ ]:
def make_step_schedule(n_rounds, n_steps, final_value=1.0):
    """Create a step-function schedule that increases in equal steps.

    Returns array of length n_rounds with values from 0 to final_value.
    Example: n_steps=4, n_rounds=100 -> jumps at rounds 12, 25, 37, 50
    """
    schedule = np.zeros(n_rounds)
    step_size = final_value / n_steps
    step_interval = n_rounds // (2 * n_steps)
    for i in range(n_steps):
        start_round = (i + 1) * step_interval
        schedule[start_round:] = (i + 1) * step_size
    return schedule


def make_oneshot_schedule(n_rounds, switch_round=50, value=1.0):
    """Create a one-shot transition: 0 until switch_round, then value."""
    schedule = np.zeros(n_rounds)
    schedule[switch_round:] = value
    return schedule


def make_decay_schedule(n_rounds, initial=1.0, tau=20.0):
    """Create an exponential decay schedule: initial * exp(-t/tau)."""
    t = np.arange(n_rounds)
    return initial * np.exp(-t / tau)


def make_persistent_schedule(n_rounds, value=1.0):
    """Constant value from round 0."""
    return np.full(n_rounds, value)

In [ ]:
n_rounds = 100

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(make_persistent_schedule(n_rounds), label='Persistent (GG)', linewidth=2)
ax.plot(make_step_schedule(n_rounds, 4), label='4-step', linewidth=2)
ax.plot(make_step_schedule(n_rounds, 8), label='8-step', linewidth=2)
ax.plot(make_oneshot_schedule(n_rounds), label='One-shot (OS)', linewidth=2)
ax.plot(make_decay_schedule(n_rounds), label='Exponential decay', linewidth=2, linestyle='--')
ax.set_xlabel('Sampling round')
ax.set_ylabel('Constraint strength c(t)')
ax.set_title('Constraint Schedule Strategies')
ax.legend()
plt.tight_layout()
plt.show()

## Base Model + Time-Varying Sampling

The key pattern: at each round, rebuild the model with the current constraint
strength from the schedule, then sample. This is the general approach for any
time-varying potential in THRML.

In [ ]:
def build_drift_ising(K=16, c_A=0.0, c_B=0.0):
    """Build an Ising EBM with tunable constraint strength.

    Same model as 03_drift_cascade_ebm — see that notebook for derivation.
    """
    nodes_A = [SpinNode() for _ in range(K)]
    nodes_B = [SpinNode() for _ in range(K)]
    all_nodes = nodes_A + nodes_B

    theta_star = 0.85
    theta_gg = 0.06
    eps = 1e-6

    b_drift = 0.5 * np.log(theta_star / (1 - theta_star))
    b_gg_net = 0.5 * np.log(max(theta_gg, eps) / max(1 - theta_gg, eps))
    b_constraint_full = b_drift - b_gg_net

    biases_A = np.full(K, b_drift - b_constraint_full * c_A)
    biases_B = np.full(K, b_drift - b_constraint_full * c_B)
    biases = jnp.array(np.concatenate([biases_A, biases_B]))

    edges = []
    weights = []

    J_within = 0.02 / K
    for i in range(K):
        for j in range(i + 1, K):
            edges.append((nodes_A[i], nodes_A[j]))
            weights.append(J_within)
    for i in range(K):
        for j in range(i + 1, K):
            edges.append((nodes_B[i], nodes_B[j]))
            weights.append(J_within)

    coupling_strength = 0.15
    J_cross = coupling_strength / (K * K)
    for i in range(K):
        for j in range(K):
            edges.append((nodes_A[i], nodes_B[j]))
            weights.append(J_cross)

    weights = jnp.array(np.array(weights))
    beta = jnp.array(1.0)

    ebm = IsingEBM(all_nodes, edges, biases, weights, beta)
    return ebm, nodes_A, nodes_B, all_nodes


def sample_with_schedule(schedule, K=16, n_samples_per_round=10, seed=42):
    """Run THRML sampling with per-round bias updates.

    At each round, rebuild the model with the current constraint strength
    from the schedule, then take a short sampling burst.
    """
    n_rounds = len(schedule)
    trajectory = []

    for r in range(n_rounds):
        c = schedule[r]
        ebm, nodes_A, nodes_B, all_nodes = build_drift_ising(K=K, c_A=c, c_B=0.0)

        blocks = [Block([node]) for node in all_nodes]
        program = IsingSamplingProgram(ebm=ebm, free_blocks=blocks, clamped_blocks=[])
        sched = SamplingSchedule(n_warmup=50, n_samples=n_samples_per_round, steps_per_sample=5)

        init_state = [jnp.array([False]) for _ in all_nodes]
        key = jax.random.PRNGKey(seed + r)

        samples = sample_states(
            key=key, program=program, schedule=sched,
            init_state_free=init_state, state_clamp=[],
            nodes_to_sample=blocks,
        )

        # Extract mean theta for agent A
        samples_A = jnp.stack([s[:, 0] for s in samples[:K]], axis=-1)
        theta_A = float(jnp.mean(samples_A.astype(jnp.float32)))
        trajectory.append(theta_A)

    return np.array(trajectory)

## Step-Function Results

Compare 4-step and 8-step schedules. Does doubling the injection steps improve convergence?

In [ ]:
n_rounds = 100

sched_4 = make_step_schedule(n_rounds, 4)
sched_8 = make_step_schedule(n_rounds, 8)

print("Sampling 4-step schedule...")
traj_4 = sample_with_schedule(sched_4, K=16, seed=42)
print("Sampling 8-step schedule...")
traj_8 = sample_with_schedule(sched_8, K=16, seed=43)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ax1.plot(traj_4, label='4-step', linewidth=2, color='steelblue')
ax1.plot(traj_8, label='8-step', linewidth=2, color='coral')
ax1.set_ylabel('θ (drift state)')
ax1.set_title('Drift Trajectory: 4-step vs 8-step')
ax1.legend()

ax2.plot(sched_4, label='4-step schedule', linewidth=2, color='steelblue', alpha=0.7)
ax2.plot(sched_8, label='8-step schedule', linewidth=2, color='coral', alpha=0.7)
ax2.set_ylabel('Constraint c(t)')
ax2.set_xlabel('Round')
ax2.legend()
plt.tight_layout()
plt.show()

print(f"\nFinal θ — 4-step: {traj_4[-1]:.3f}, 8-step: {traj_8[-1]:.3f}")
print(f"8-step {'improves on' if traj_8[-1] < traj_4[-1] else 'does NOT improve on'} 4-step")

## One-Shot vs Iterative

The one-shot schedule produces a sharp drop at injection, followed by partial rebound.
The iterative schedule avoids rebound but converges slower initially.

In [ ]:
sched_os = make_oneshot_schedule(n_rounds, switch_round=50)

print("Sampling one-shot schedule...")
traj_os = sample_with_schedule(sched_os, K=16, seed=44)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ax1.plot(traj_4, label='4-step', linewidth=2, color='steelblue')
ax1.plot(traj_os, label='One-shot', linewidth=2, color='coral')
ax1.set_ylabel('θ (drift state)')
ax1.set_title('One-Shot vs Iterative Injection')
ax1.axvline(x=50, color='gray', linestyle=':', alpha=0.5, label='OS injection point')
ax1.legend()

ax2.plot(sched_4, label='4-step schedule', linewidth=2, color='steelblue', alpha=0.7)
ax2.plot(sched_os, label='One-shot schedule', linewidth=2, color='coral', alpha=0.7)
ax2.set_ylabel('Constraint c(t)')
ax2.set_xlabel('Round')
ax2.legend()
plt.tight_layout()
plt.show()

## Persistent Dominates

The key result: persistent constraint from round 1 (GG) massively outperforms
every iterative schedule. If you can apply the constraint from the start, do it.

In [ ]:
schedules = {
    'Persistent (GG)': make_persistent_schedule(n_rounds),
    '4-step': sched_4,
    '8-step': sched_8,
    'One-shot': sched_os,
    'Unconstrained': np.zeros(n_rounds),
}

trajectories = {
    '4-step': traj_4,
    '8-step': traj_8,
    'One-shot': traj_os,
}

# Sample the remaining schedules
print("Sampling persistent schedule...")
trajectories['Persistent (GG)'] = sample_with_schedule(schedules['Persistent (GG)'], K=16, seed=45)
print("Sampling unconstrained...")
trajectories['Unconstrained'] = sample_with_schedule(schedules['Unconstrained'], K=16, seed=46)

# All trajectories overlaid
colors = {'Persistent (GG)': 'green', '4-step': 'steelblue', '8-step': 'coral',
          'One-shot': 'orange', 'Unconstrained': 'red'}

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
for name in ['Unconstrained', 'One-shot', '4-step', '8-step', 'Persistent (GG)']:
    ax1.plot(trajectories[name], label=name, linewidth=2, color=colors[name])
    ax2.plot(schedules[name], label=name, linewidth=2, color=colors[name], alpha=0.7)

ax1.set_ylabel('θ (drift state)')
ax1.set_title('Drift Trajectories Under Different Schedules')
ax1.legend()
ax2.set_ylabel('Constraint c(t)')
ax2.set_xlabel('Round')
ax2.set_title('Applied Constraint Schedules')
plt.tight_layout()
plt.show()

## Quantitative Comparison

In [ ]:
print(f"{'Schedule':<20} {'Final θ':>10} {'Mean θ':>10} {'Std θ':>10}")
print("-" * 55)
for name in ['Persistent (GG)', '4-step', '8-step', 'One-shot', 'Unconstrained']:
    traj = trajectories[name]
    print(f"{name:<20} {traj[-1]:>10.3f} {np.mean(traj):>10.3f} {np.std(traj):>10.3f}")

print(f"\nOrdering: GG << IT-4 ≈ IT-8 < OS < U")

## Key Takeaways

1. **Persistent beats iterative.** If you can apply the full potential from the start,
   do it. No annealing schedule outperforms constant bias in this system.

2. **More steps ≠ better.** 8-step does not reliably outperform 4-step. The marginal
   value of each additional injection step decreases — a diminishing-returns pattern
   consistent with nonlinear response to bias changes.

3. **One-shot rebounds.** Sudden potential changes produce transient overshooting
   followed by partial recovery. The system needs time to re-equilibrate after
   a large perturbation.

4. **Schedule shape > step count.** A front-loaded schedule (large early steps,
   small later steps) outperforms uniform spacing. The response function is
   state-dependent and nonlinear.

## When to Use Time-Varying Potentials

- **Simulated annealing:** Gradually reduce temperature to find low-energy states
- **Constraint scheduling:** Inject bias over time when full constraint isn't available from the start
- **Curriculum learning:** Increase model complexity over training by changing the energy landscape
- **Non-equilibrium sampling:** Study driven systems by varying external fields

## References

[1] Kirkpatrick et al. (1983). Optimization by Simulated Annealing. *Science*.  
[2] Morris, A. (2025). The Architecture of Drift. Zenodo. https://doi.org/10.5281/zenodo.14793653